# 04. else and finally

**Tier:** Intermediate. **Why you'd use this:** You parse a payment amount, log success only when parsing worked, and always close the audit line afterward, even when parsing failed.

**After this lesson you can:** put success-only work in `else`; put cleanup in `finally`; say the order when both run.
About 25 minutes. **Uses:** [Tracebacks](03-tracebacks-lesson.ipynb)


## Quick review

1. Where do you start reading a traceback?
2. Name two of the five `pdb` commands from the last lesson.

<details><summary>Answers</summary>

The last line, then the arrow. Any two of `l`, `n`, `s`, `p`, `c`.
</details>


## The idea

`else` on a `try` runs when no exception was raised in the `try` block. `finally` runs whether the `try` worked, failed, or returned. Use `else` for success-only follow-up, such as logging a good parse. Use `finally` for cleanup you must not skip, such as closing an audit line.

<details><summary>Explain more</summary>

The order is `try`, then either `except` or `else`, then `finally`. A `return` inside `try` still runs `finally` before the value leaves. That is why cleanup belongs in `finally`, not after the whole statement. Success work in `finally` is a lie waiting to happen on the failure path.
</details>


## See it work

Parse an amount, log success, and always mark the audit done.


In [1]:
def audit_amount(text):
    log = []
    # 1. Try the parse.
    try:
        amount = int(text)
    except ValueError:
        log.append("bad")
    else:
        # 2. Only on success.
        log.append("ok:" + str(amount))
    finally:
        # 3. Always.
        log.append("done")
    return log


print(audit_amount("20"))
print(audit_amount("x"))


['ok:20', 'done']
['bad', 'done']


## Check yourself

Answer these before you run the cell.

1. What does `audit_amount("20")` return?
2. If `else` appended after a failed parse, what bug would you get?


In [2]:
def audit_amount(text):
    log = []
    try:
        amount = int(text)
    except ValueError:
        log.append("bad")
    else:
        log.append("ok:" + str(amount))
    finally:
        log.append("done")
    return log


print(audit_amount("20"))
print(audit_amount("x"))


['ok:20', 'done']
['bad', 'done']


<details><summary>Answers</summary>

Success returns `['ok:20', 'done']`. Failure returns `['bad', 'done']`. Code in `else` that assumed `amount` exists would raise `UnboundLocalError` on the failure path if you had put it in `finally` by mistake.
</details>


## Why it works this way

`else` exists so success follow-up is not mixed into the `try` block. Keeping the `try` small makes it obvious which line can raise. Anything in `else` is not covered by the same `except`, which is what you want for code that should not fail the same way. `finally` exists because cleanup must run even when the function returns or raises. The interpreter schedules `finally` before the frame really leaves. Deep dive: [Memory management](../../internals/11-memory/01-refcount-lesson.ipynb).

Order is the contract: handle or celebrate, then clean up. Putting cleanup after the whole `try`/`except` skips it when you `return` inside `except`. Putting success work in `finally` runs it on failure too, which is how you log `"ok"` after a bad parse. The evidence cell shows `"finally"` appended even after `return` inside `try` or `except`.

The cost of `finally` is a guaranteed visit. That is what you want for files, locks, and audit marks. When you later learn `with`, it is this same guarantee packaged as a protocol. Until then, write the `finally` yourself whenever a side effect must not be skipped.

A second rule of thumb: if a line can raise the same exception type as the `try`, keep it out of `else` unless you want that failure to escape. `else` is for work that assumes the `try` already succeeded, not for a second risky step you forgot to guard.

That is also why the success log includes the parsed amount. The name `amount` exists on the `else` path. On the `except` path it was never bound. Mixing those paths in one block is how silent audit bugs and `UnboundLocalError` both appear.


In [3]:
def order(text):
    log = []
    try:
        int(text)
        log.append("try")
        return log
    except ValueError:
        log.append("except")
        return log
    finally:
        log.append("finally")


print(order("1"))
print(order("x"))


['try', 'finally']
['except', 'finally']


## When to use it

Use `else` for code that should run only after a clean `try`. Use `finally` for cleanup. Prefer `with` for files when you learn it, because it packages the `finally`. Don't use `else` for cleanup. Don't put the risky line in `else`. Don't put success logging in `finally`.


## Common mistake

Logging success in `finally`. Failure still appends `"ok"`, so the audit lies.


In [4]:
def audit_wrong(text):
    log = []
    try:
        int(text)
    except ValueError:
        log.append("bad")
    finally:
        log.append("ok")
        log.append("done")
    return log


print(audit_wrong("x"))


['bad', 'ok', 'done']


Keep success in `else`. Keep cleanup in `finally`.


In [5]:
def audit_amount(text):
    log = []
    try:
        amount = int(text)
    except ValueError:
        log.append("bad")
    else:
        log.append("ok:" + str(amount))
    finally:
        log.append("done")
    return log


print(audit_amount("x"))


['bad', 'done']


## Try it

Change `audit_amount` so a successful parse also appends `"charged"` after `"ok:..."`. Which clause do you extend, and what would go wrong if you appended that mark in `finally`?


## Recap

- `else` runs only when `try` did not raise.
- `finally` always runs, even after `return`.
- Success logging belongs in `else`. Cleanup belongs in `finally`.

**Next:** [practice](04-else-finally-practice.ipynb)
